# Frozen AraBEST Mixture All on Kaggle — 2,000 vs. 4,000 clips

The 4,000-clip training split contains the exact 2,000 baseline training rows and **100 new clips for each of the 20 dialects**. Both arms use frozen AraBEST-RQ-300M, the same Mixture All head, matched seeds, and 1,000 optimizer updates per seed. Each arm fits z-score statistics on its own training split. Both are evaluated on the same 200 ADI-20 dev rows and 6,819 Casablanca validation rows.

1. Import this notebook into a private Kaggle Notebook. Turn on **Internet** and choose **GPU T4 ×2**. Feature extraction uses one GPU. A CPU session also works if the complete feature caches are attached.
2. Upload **ADI20_4K_Training_Manifest.zip** as a private Kaggle Dataset and attach it. The notebook accepts either the ZIP or the extracted **train_scaling_4000.jsonl**. No full train manifest or raw audio upload is needed.
3. Attach the saved outputs of your **500 vs. 2,000** notebook if they contain **reusable_adi_cache**. Also attach any earlier notebook outputs or feature datasets used by that run. Kaggle saves new output files from each run, so prior inputs may still be needed for full cache coverage.
4. Run through **Check cache coverage**. If the 2K training and both evaluation caches are complete, only the 2,000 new training clips need encoding. Set **PROBE_ONLY = True** to time up to 200 of the missing training clips. Set it back to **False** and rerun the extraction cell to continue in the same session.
5. Download **adi20_2k_vs_4k_results.zip** from Output. Use **paired_deltas_percentage_points.csv** for the seed-by-seed 4K minus 2K comparison.

Extraction stops between batches or shards at a **6-hour cap per run**. A network download already in progress can extend beyond the cap. If the run pauses, save its output and attach it on the next run. If sparse ADI-20 reads fail, set **SPARSE_HF_READS = False** to download one shard at a time. For authenticated Hugging Face access, add the **HF_TOKEN** Kaggle secret.

Your previous 2K run reported **84.83% ADI-20 dev accuracy** and **55.61% Casablanca accuracy / 60.14% macro-F1**. Check Casablanca macro-F1 and Jordanian recall and precision alongside accuracy. The NADI blind test is outside this scaling experiment.

## 1. Set up Kaggle and load the pinned splits

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = Path("/tmp/adi-mixture-4k")
COMMIT = "b64437a37e7eda2f1721a56860278640bf085bdd"
REPO_URL = "https://github.com/alizbeeb5/Arabic-Dialect-Identification.git"

SEEDS = [490, 491, 492]
UPDATES = 1000                 # Same optimizer updates for both training sizes.
PROBE_ONLY = False             # True extracts up to 200 missing training clips and exits cleanly.
PROBE_CLIPS = 200
MAX_EXTRACT_HOURS = 6.0        # Per run, checked between shards and encode batches.
EXTRACT_BATCH = 4
EXTRACT_CHUNK = 48
SPARSE_HF_READS = True         # False downloads each needed ADI-20 shard in full.

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
WORK.mkdir(parents=True, exist_ok=True)

def git(*args):
    subprocess.run(["git", "-C", str(REPO), *args], check=True)

if not (REPO / ".git").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    git("init", "--quiet")
    git("remote", "add", "origin", REPO_URL)

pinned_commit = subprocess.run(
    ["git", "-C", str(REPO), "cat-file", "-e", f"{COMMIT}^{{commit}}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
if pinned_commit.returncode:
    git("fetch", "--quiet", "--depth=1", "origin", COMMIT)
git("checkout", "--quiet", "--detach", COMMIT)
sys.path.insert(0, str(REPO))

# Keep Kaggle's CUDA build of PyTorch and torchaudio.
packages = {"soundfile": "soundfile", "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub"}
missing_packages = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)

In [ ]:
import hashlib
import json
import shutil
import time
from zipfile import ZipFile
from collections import Counter, defaultdict
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score

from src.models import acoustic_encoder as ae
from src.experiments import acoustic_classifier as clf, layer_mixture as lm
from src.experiments import mfcc_svm_baseline as base
from src.experiments.casablanca_check import diagnostics

HEAD_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
torch.backends.cuda.matmul.allow_tf32 = False

M = REPO / "data/manifests"
RUN = WORK / "scaling_2k_vs_4k_results" / time.strftime("%Y%m%d-%H%M%S")
CONFIG = json.loads((REPO / "configs/layer_mixture.json").read_text())
LABELS = json.loads((REPO / "configs/labels.json").read_text())["label_order"]
LABEL_ID = {label: i for i, label in enumerate(LABELS)}
JOR = LABEL_ID["JOR"]
OPT = CONFIG["optimization"]
ADI_REV = "c8da454c9f474bb075d3fe608940ee29e8ccfa43"
CAS_REV = "8951b1b88e28c1107142ced57967b8d16350951d"
MANIFESTS = {
    "train_scaling_2000": "41514213e14bfbd82d4fbf8cfe6eae04a2faf4f878509a15758f431dc41f7929",
    "train_scaling_4000": "ebeb36edce3399a3b3eb116ae3158185e1c9e6bc0308f270b3c45d9d6901f120",
    "dev_pilot": "fb8684c35ef68e25834d5a4c82cddbd31044a899a63b665c41d3f9a179052d02",
    "casablanca_validation": "5ed7a36491ba6a41782302e1f3d31c3033b4d72ac18f1f7a2d04e85177dd064e",
}

def scaling_manifest():
    name = "train_scaling_4000.jsonl"
    for path in sorted(INPUT.rglob(name)):
        if path.is_file():
            raw = path.read_bytes()
            if hashlib.sha256(raw).hexdigest() == MANIFESTS["train_scaling_4000"]:
                return raw
    for package in sorted(INPUT.rglob("ADI20_4K_Training_Manifest.zip")):
        with ZipFile(package) as archive:
            raw = archive.read(name)
        if hashlib.sha256(raw).hexdigest() == MANIFESTS["train_scaling_4000"]:
            return raw
    raise FileNotFoundError("Attach the supplied ADI20_4K_Training_Manifest.zip or its extracted manifest.")

def read_rows(name):
    raw = scaling_manifest() if name == "train_scaling_4000" else (M / f"{name}.jsonl").read_bytes()
    assert hashlib.sha256(raw).hexdigest() == MANIFESTS[name], f"Manifest mismatch: {name}"
    return [json.loads(line) for line in raw.splitlines() if line.strip()]

train2000 = read_rows("train_scaling_2000")
train4000 = read_rows("train_scaling_4000")
dev = read_rows("dev_pilot")
cas = read_rows("casablanca_validation")

for rows, per_label in ((train2000, 100), (train4000, 200)):
    assert Counter(row["label"] for row in rows) == {label: per_label for label in LABELS}
    assert all(row["split"] == "train" and row["source_revision"] == ADI_REV for row in rows)
    assert len({row["clip_id"] for row in rows}) == len(rows)

by_id = {row["clip_id"]: row for row in train4000}
base_ids = {row["clip_id"] for row in train2000}
assert all(by_id[row["clip_id"]] == row for row in train2000), "2K training rows changed in the 4K split"
added = [row for row in train4000 if row["clip_id"] not in base_ids]
assert Counter(row["label"] for row in added) == {label: 100 for label in LABELS}
for field in ("clip_id", "audio_sha256", "pcm_sha256"):
    training_values = {row[field] for row in train4000}
    eval_values = {row[field] for row in dev + cas if row.get(field)}
    assert len(training_values) == 4000
    assert training_values.isdisjoint(eval_values), f"Training/evaluation overlap by {field}"
assert len(dev) == 200 and len(cas) == 6819

print("Train: 2,000 and 4,000 nested clips. Eval: 200 ADI-20 dev and 6,819 Casablanca validation rows.")


## 2. Check cache coverage

In [ ]:
feature_config = clf.feature_config(CONFIG["encoder"])
provenance = lm.expected_provenance(feature_config)
write_cache = ae.FeatureCache(WORK / "reusable_adi_cache", provenance)
cas_ok = [row for row in cas if row.get("preparation_status", "ok") == "ok"]
assert len(cas_ok) == 6818

# Use attached caches in place. New parts go only into the writable cache.
read_caches = []
for config_path in sorted(INPUT.rglob("config.json")):
    index_path = config_path.with_name("index.jsonl")
    if config_path.parent.name != write_cache.root.name or not index_path.is_file():
        continue
    mounted = ae.FeatureCache(WORK, provenance)
    mounted.root, mounted.index_path = config_path.parent, index_path
    read_caches.append(mounted)

def feature_locations(rows):
    indices = [(cache.root, cache.index()) for cache in [*read_caches, write_cache]]
    locations = {}
    part_exists = {}
    for row in rows:
        clip_id = row["clip_id"]
        key = ae.clip_cache_key(clip_id, row["audio_sha256"], provenance)
        for root, index in indices:
            saved = index.get(clip_id)
            if saved is None or saved["cache_key"] != key:
                continue
            part = root / saved["part"]
            if part not in part_exists:
                part_exists[part] = part.is_file()
            if part_exists[part]:
                locations[clip_id] = (part, saved["row"], key)
                break
    return locations

def missing_features(rows):
    locations = feature_locations(rows)
    return [row for row in rows if row["clip_id"] not in locations]

def feature_batches(rows, batch_size=64):
    locations = feature_locations(rows)
    groups = defaultdict(list)
    for i, row in enumerate(rows):
        clip_id = row["clip_id"]
        part, position, key = locations[clip_id]
        groups[part].append((i, position, clip_id, key))
    for part, entries in groups.items():
        with np.load(part, allow_pickle=False) as data:
            values, ids, keys = data["features"], data["clip_ids"], data["cache_keys"]
            for start in range(0, len(entries), batch_size):
                batch = entries[start:start + batch_size]
                for _, position, clip_id, key in batch:
                    assert ids[position] == clip_id and keys[position] == key, f"Invalid cache row for {clip_id}"
                features = values[[item[1] for item in batch]].astype(np.float32, copy=False)
                assert np.isfinite(features).all(), f"Nonfinite features in {part}"
                yield [item[0] for item in batch], features

def load_features(rows):
    result = np.empty((len(rows), 26, 848), np.float32)
    for positions, values in feature_batches(rows):
        result[positions] = values
    return result

print("Attached feature caches:", len(read_caches))
for name, rows in (("ADI-20 train 2,000", train2000), ("New ADI-20 training clips", added), ("ADI-20 dev", dev), ("Casablanca", cas_ok)):
    missing = len(missing_features(rows))
    print(f"{name}: {len(rows) - missing}/{len(rows)} cached")


## 3. Encode only missing clips

In [ ]:
from importlib.metadata import PackageNotFoundError, version

encoder = None

def get_encoder():
    global encoder
    if encoder is None:
        try:
            installed = version("speechbrain")
        except PackageNotFoundError:
            installed = None
        if installed != "1.1.2":
            subprocess.run([
                sys.executable, "-m", "pip", "install", "-q", "--no-deps", "speechbrain==1.1.2",
            ], check=True)
        optional = {"hyperpyyaml": "hyperpyyaml", "sentencepiece": "sentencepiece", "joblib": "joblib"}
        needed = [package for module, package in optional.items() if importlib.util.find_spec(module) is None]
        if needed:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)
        encoder = ae.AcousticEncoder.from_pretrained(
            feature_config, device="cuda:0", cache_dir="/tmp/adi-hf-model"
        )
        clf.frozen_summary(encoder)
        assert json.loads(json.dumps(encoder.provenance())) == provenance, "Encoder provenance mismatch"
    return encoder

def optional_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return False

def encode_shard(handle, rows, scratch, label, deadline, on_encoded):
    import pyarrow.parquet as pq

    wanted = {int(row["source_row"]): row for row in rows}
    assert len(wanted) == len(rows), "Duplicate row in the pinned manifest"
    with pq.ParquetFile(handle) as parquet:
        offset = 0
        for group in range(parquet.num_row_groups):
            end = offset + parquet.metadata.row_group(group).num_rows
            selected = [i for i in wanted if offset <= i < end]
            if selected:
                position = offset
                for batch in parquet.iter_batches(
                    row_groups=[group], columns=["audio"], batch_size=EXTRACT_CHUNK
                ):
                    if time.monotonic() >= deadline:
                        return False
                    batch_end = position + batch.num_rows
                    block = []
                    for i in selected:
                        if position <= i < batch_end:
                            row = wanted.pop(i)
                            payload = batch.column(0).field("bytes")[i - position].as_py()
                            if not payload or hashlib.sha256(payload).hexdigest() != row["audio_sha256"]:
                                raise RuntimeError(f"Audio SHA256 mismatch for {row['clip_id']}")
                            path = ae.resolve_audio_path(scratch, row["audio_path"])
                            path.parent.mkdir(parents=True, exist_ok=True)
                            path.write_bytes(payload)
                            block.append(row)
                    if block:
                        _, failures = clf.prepare_features(
                            get_encoder(), write_cache, block, scratch,
                            {"batch_size": EXTRACT_BATCH, "chunk_size": EXTRACT_CHUNK}, label,
                        )
                        if failures:
                            with (WORK / "extraction_failures.jsonl").open("a", encoding="utf-8") as log:
                                for failure in failures:
                                    log.write(json.dumps(failure) + "\n")
                            raise RuntimeError(f"{len(failures)} failed features. See extraction_failures.jsonl.")
                        for row in block:
                            (scratch / row["audio_path"]).unlink()
                        on_encoded(len(block))
                    position = batch_end
                    if position > max(selected):
                        break
            offset = end
            if not wanted:
                break
    if wanted:
        raise RuntimeError(f"Pinned parquet did not contain {len(wanted)} requested rows.")
    return True

def fill_features(rows, label, deadline, limit=None):
    missing = sorted(
        missing_features(rows),
        key=lambda row: (row["source_dataset"], row["source_shard"], row["source_row"]),
    )
    if not missing:
        print(f"{label}: reusing all {len(rows)} cached clips")
        return True
    if not torch.cuda.is_available():
        raise RuntimeError("Select GPU T4 x2 or attach feature caches for the missing rows.")

    from huggingface_hub import HfFileSystem, hf_hub_download

    target = missing if limit is None else missing[:limit]
    buckets = defaultdict(list)
    for row in target:
        source, revision, shard = row["source_dataset"], row["source_revision"], row["source_shard"]
        if source == "UBC-NLP/Casablanca":
            assert revision == CAS_REV and "validation" in shard.lower() and "test" not in shard.lower()
        else:
            assert source == "UBC-NLP/NADI_2026_ADI20_micro" and revision == ADI_REV
        buckets[source, revision, shard].append(row)

    token = optional_hf_token()
    hub = HfFileSystem(token=token)
    started = time.monotonic()
    encoded = 0
    probe_size = min(PROBE_CLIPS, len(target))

    def record(count):
        nonlocal encoded
        previous = encoded
        encoded += count
        if previous < probe_size <= encoded:
            elapsed = time.monotonic() - started
            remaining_hours = elapsed * (len(missing) - encoded) / encoded / 3600
            print(
                f"{label}: {encoded} newly encoded in {elapsed / 60:.1f} min. "
                f"Rough estimate for remaining {len(missing) - encoded}: {remaining_hours:.2f} h.",
                flush=True,
            )

    for i, ((source, revision, shard), clips) in enumerate(buckets.items(), 1):
        if time.monotonic() >= deadline:
            return False
        print(f"{label}: shard {i}/{len(buckets)} | {len(clips)} clips | {shard}", flush=True)
        with TemporaryDirectory(prefix="adi-shard-") as directory:
            path = Path(directory)
            scratch = path / "audio"
            scratch.mkdir()
            if SPARSE_HF_READS and source == "UBC-NLP/NADI_2026_ADI20_micro":
                with hub.open(f"datasets/{source}@{revision}/{shard}", "rb", block_size=8 * 1024**2) as handle:
                    finished = encode_shard(handle, clips, scratch, label, deadline, record)
            else:
                local = hf_hub_download(
                    repo_id=source, repo_type="dataset", revision=revision, filename=shard,
                    local_dir=path / "parquet", token=token,
                )
                finished = encode_shard(local, clips, scratch, label, deadline, record)
            if not finished:
                return False
    assert not missing_features(target), f"{label}: feature cache is incomplete"
    return len(target) == len(missing)

In [ ]:
assert PROBE_CLIPS > 0 and MAX_EXTRACT_HOURS > 0
deadline = time.monotonic() + MAX_EXTRACT_HOURS * 3600
if PROBE_ONLY:
    fill_features(added, "New ADI-20 training timing probe", deadline, limit=PROBE_CLIPS)
    ready = False
    print("Probe finished. Set PROBE_ONLY = False to continue from the saved cache.")
else:
    ready = fill_features(train4000 + dev, "ADI-20 train and dev", deadline)
    if ready:
        ready = fill_features(cas_ok, "Casablanca validation", deadline)
    if not ready:
        print("Extraction budget reached. Save this run's output and attach it to resume in a new session.")

encoder = None
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Writable feature cache:", write_cache.root)

## 4. Train both heads with the same update budget

In [ ]:
def fit_head(features, labels, seed):
    init_seed, shuffle_seed = clf.run_seeds(seed)
    model = lm.build_model(tuple(features.shape), len(LABELS), init_seed).to(HEAD_DEVICE)
    optimizer = lm.make_optimizer(model, OPT)
    features, labels = features.to(HEAD_DEVICE), labels.to(HEAD_DEVICE)
    generator = torch.Generator().manual_seed(shuffle_seed)
    steps = 0
    while steps < UPDATES:
        order = torch.randperm(len(features), generator=generator)
        for indices in order.split(OPT["batch_size"]):
            indices = indices.to(HEAD_DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(features[indices]), labels[indices])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), OPT["gradient_clip_norm"])
            optimizer.step()
            steps += 1
            if steps == UPDATES:
                break
    return model.cpu().eval()

TRAIN_ARMS = {"train_2000": train2000, "train_4000": train4000}
models = {}
normalization = {}
if ready:
    RUN.mkdir(parents=True, exist_ok=True)
    for arm, rows in TRAIN_ARMS.items():
        raw = load_features(rows)
        mean = raw.mean(axis=0)
        std = np.maximum(raw.std(axis=0), CONFIG["preprocessing"]["std_floor"])
        normalization[arm] = mean, std
        raw -= mean
        raw /= std
        features = torch.from_numpy(raw)
        labels = torch.tensor([LABEL_ID[row["label"]] for row in rows], dtype=torch.long)
        arm_dir = RUN / arm
        arm_dir.mkdir(exist_ok=True)
        np.savez(arm_dir / "zscore_stats.npz", mean=mean, std=std,
                 layer_names=np.array(provenance["layer_names"]))
        for seed in SEEDS:
            model = fit_head(features, labels, seed)
            models[arm, seed] = model
            torch.save(
                {"state_dict": model.state_dict(), "labels": LABELS, "layers": provenance["layer_names"],
                 "seed": seed, "updates": UPDATES, "train_size": len(rows), "config": CONFIG},
                arm_dir / f"head_{seed}.pt",
            )
            np.save(arm_dir / f"layer_weights_{seed}.npy", model.layer_weights().detach().numpy())
            print(f"{arm}, seed {seed}: trained for {UPDATES} updates", flush=True)
        del features, raw
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Evaluate both splits and export results

In [ ]:
@torch.no_grad()
def predict_split(readable, all_rows):
    valid_positions = [i for i, row in enumerate(all_rows) if row.get("preparation_status", "ok") == "ok"]
    assert len(valid_positions) == len(readable)
    predictions = {key: np.full(len(all_rows), -1, dtype=int) for key in models}
    for indices, batch in feature_batches(readable, batch_size=128):
        output_positions = [valid_positions[i] for i in indices]
        for arm, (mean, std) in normalization.items():
            features = torch.from_numpy((batch - mean) / std)
            for seed in SEEDS:
                predictions[arm, seed][output_positions] = models[arm, seed](features).argmax(1).numpy()
    for values in predictions.values():
        assert (values[valid_positions] >= 0).all(), "Missing prediction on readable audio"
    return predictions

def score(rows, pred):
    gold = np.array([LABEL_ID[row["label"]] for row in rows])
    present = np.unique(gold)
    jordanian = gold == JOR
    predicted_jor = pred == JOR
    correct = gold == pred
    names = [LABELS[i] if i >= 0 else "FAILURE" for i in pred]
    region = diagnostics(rows, [name if name != "FAILURE" else None for name in names],
                         [LABELS[i] for i in present], LABELS)[0]["region_accuracy"]
    metrics = {
        "accuracy": correct.mean(),
        "macro_f1": f1_score(gold, pred, labels=present, average="macro", zero_division=0),
        "jor_recall": correct[jordanian].mean(),
        "jor_precision": correct[predicted_jor].mean() if predicted_jor.any() else 0.0,
        "non_jor_accuracy": correct[~jordanian].mean(),
        "region_accuracy": region,
        "n": len(gold),
        "failures": int((pred < 0).sum()),
    }
    return gold, names, metrics

if ready:
    records = []
    for split, readable, rows in (("ADI20_dev", dev, dev), ("Casablanca", cas_ok, cas)):
        predictions = predict_split(readable, rows)
        for (arm, seed), pred in predictions.items():
            gold, names, metrics = score(rows, pred)
            records.append({"arm": arm, "seed": seed, "split": split,
                            "train_size": len(TRAIN_ARMS[arm]), **metrics})
            folder = RUN / arm / str(seed)
            folder.mkdir(exist_ok=True)
            pd.DataFrame({"clip_id": [row["clip_id"] for row in rows],
                          "gold": [LABELS[i] for i in gold], "prediction": names}).to_csv(
                folder / f"{split}_predictions.csv", index=False,
            )
            labels = LABELS + ["FAILURE"]
            cm = confusion_matrix(gold, np.where(pred >= 0, pred, len(LABELS)), labels=range(len(labels)))
            pd.DataFrame(cm, index=labels, columns=labels).to_csv(folder / f"{split}_confusion.csv")

    results = pd.DataFrame(records)
    metrics = ["accuracy", "macro_f1", "jor_recall", "jor_precision", "non_jor_accuracy", "region_accuracy"]
    summary = results.groupby(["split", "arm"])[metrics].agg(["mean", "std"])
    baseline = results[results.arm == "train_2000"].set_index(["split", "seed"])
    scaled = results[results.arm == "train_4000"].set_index(["split", "seed"])
    deltas = 100 * (scaled[metrics] - baseline[metrics])
    results.to_csv(RUN / "results.csv", index=False)
    summary.to_csv(RUN / "summary.csv")
    deltas.to_csv(RUN / "paired_deltas_percentage_points.csv")
    display((100 * summary).round(2))
    print("4,000 minus 2,000 clips, mean percentage points across matched seeds")
    display(deltas.groupby(level="split").mean().round(2))

    protocol = {
        "repo_commit": COMMIT, "encoder": CONFIG["encoder"], "manifest_sha256": MANIFESTS,
        "seeds": SEEDS, "optimizer_updates_per_head": UPDATES, "head_device": HEAD_DEVICE,
        "normalization": "Fit independently on the training clips of each arm.",
        "previous_2k_reference": {"adi20_dev_accuracy": 0.8483, "casablanca_accuracy": 0.5561, "casablanca_macro_f1": 0.6014},
        "evaluation": "Same 200 ADI-20 dev rows and 6,819 Casablanca validation rows for both arms.",
        "speaker_independence_verified": False,
    }
    (RUN / "protocol.json").write_text(json.dumps(protocol, indent=2))
    archive = shutil.make_archive(str(WORK / "adi20_2k_vs_4k_results"), "zip", RUN)
    print("Download from Kaggle Output:", archive)